# ML-07 — Baseline Action Score and Top-10 Review

**Lane:** Refresh / Content Opportunity Scoring

**Development slice:** March 2026. The decision is made at the end of March, so the baseline uses only March information.

This notebook does three things required for ML-07:
1. Check two rule-linked signals with visible bucket tables and `n`, then print one-word verdicts.
2. Freeze one transparent rule: score + one reason code + one action label, and write the ranked queue.
3. Review the top 10 with a skeptic's eye: action, why it is there, and what would make it wrong.

The baseline is deliberately hand-written and unfitted. A later Week-5 model must beat this baseline on the same decision setup.


## 0. Decision frame

**Decision:** Which content pages should an SEO/content editor review first?

**Decision moment:** End of March 2026.

**Rule idea:** prioritize pages with enough search visibility to matter and a search position where improvement may be plausible.

**Two session-linked signals:**
- **Volume / search visibility** — linked to the session's **quick-win / volume** flag logic.
- **CTR vs position** — linked to the session's **CTR-fix** logic.

**Guardrails:** `trend_direction`, `trend_pct`, future-month metrics, and label-derived fields are not used by the rule.


In [2]:
%pip -q install duckdb
import duckdb, os
import pandas as pd
from IPython.display import display

con = duckdb.connect()
MONTH = "2026-03"
REL = f"read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MONTH}/*.parquet')"

# In Colab, store HF_TOKEN in Secrets. The token is never written into this notebook.
token = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    token = token or userdata.get("HF_TOKEN")
except Exception:
    pass

if token:
    safe_token = token.replace("'", "''")
    con.execute(
        f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')"
    )
else:
    print("HF_TOKEN not found. If the dataset is gated, add HF_TOKEN in Colab Secrets and rerun this cell.")

schema = con.sql(f"DESCRIBE SELECT * FROM {REL} LIMIT 0").df()
print(schema[["column_name", "column_type"]].to_string(index=False))


HF_TOKEN not found. If the dataset is gated, add HF_TOKEN in Colab Secrets and rerun this cell.


HTTPException: HTTP Error: HTTP GET error on 'https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance/month=2026-03/data_0.parquet' (HTTP 401)

In [ ]:
# Resolve likely released warehouse column names from the actual schema.
cols = set(schema.column_name.tolist())

def pick(candidates):
    for c in candidates:
        if c in cols:
            return c
    raise KeyError(f"None of these columns were found: {candidates}")

impr = pick(["gsc_impressions", "impressions", "search_impressions"])
clicks = pick(["gsc_clicks", "clicks"]) if any(c in cols for c in ["gsc_clicks", "clicks"]) else None
pos = pick(["gsc_avg_position", "avg_position", "position"])
date_col = pick(["report_date", "date"])
client_col = pick(["client_id", "client"])
content_col = pick(["content_id", "content"])

print("Using:", {
    "impressions": impr,
    "clicks": clicks,
    "position": pos,
    "date": date_col,
    "client": client_col,
    "content": content_col,
})


## 1. Signal check — two bucket tables

The verdicts are computed from the executed March tables; they are not hard-coded in advance.

### Signal A — volume / search visibility

This is the flag-linked **quick-win / volume** signal. The table checks whether the higher-visibility buckets contain a useful pool of pages in the rule's opportunity band (position 4–20).

### Signal B — CTR vs position

This is the flag-linked **CTR-fix** signal. When clicks are available, the table shows median CTR by position bucket. A positive verdict means the observed March data supports the expected relationship that better positions have higher CTR than deeper positions.


In [ ]:
# One row per client × content item for the March decision window.
agg = f'''
SELECT
    {client_col} AS client_id,
    {content_col} AS content_id,
    SUM(COALESCE({impr}, 0)) AS impressions,
    {('SUM(COALESCE(' + clicks + ', 0))' if clicks else 'CAST(NULL AS DOUBLE)')} AS clicks,
    AVG(NULLIF({pos}, 0)) AS avg_position
FROM {REL}
WHERE {date_col} >= DATE '2026-03-01'
  AND {date_col} < DATE '2026-04-01'
GROUP BY 1, 2
'''
df = con.sql(agg).df()

df["impression_bucket"] = pd.cut(
    df["impressions"],
    [-1, 99, 299, 2_999, 29_999, float("inf")],
    labels=["<100", "100-299", "300-2,999", "3,000-29,999", "30,000+"],
)
df["position_bucket"] = pd.cut(
    df["avg_position"],
    [-float("inf"), 3, 10, 20, 50, float("inf")],
    labels=["top_3", "page_1", "striking", "page_3_5", "deep"],
)

df["opportunity_band"] = df["avg_position"].between(4, 20, inclusive="both")

signal_a = (
    df.groupby("impression_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_impressions=("impressions", "median"),
          opportunity_n=("opportunity_band", "sum"),
          opportunity_rate=("opportunity_band", "mean"),
      )
      .reset_index()
)
signal_a["opportunity_rate"] = (100 * signal_a["opportunity_rate"]).round(1)

print("SIGNAL A — volume / search visibility")
display(signal_a)

if clicks:
    df["ctr_pct"] = df["clicks"].div(df["impressions"].replace(0, pd.NA)).mul(100)
    signal_b = (
        df.dropna(subset=["position_bucket"])
          .groupby("position_bucket", observed=False)
          .agg(
              n=("content_id", "size"),
              median_impressions=("impressions", "median"),
              median_ctr_pct=("ctr_pct", "median"),
          )
          .reset_index()
    )
    signal_b["median_ctr_pct"] = signal_b["median_ctr_pct"].round(2)
else:
    signal_b = (
        df.dropna(subset=["position_bucket"])
          .groupby("position_bucket", observed=False)
          .agg(
              n=("content_id", "size"),
              median_impressions=("impressions", "median"),
          )
          .reset_index()
    )

print("SIGNAL B — position / CTR-vs-position")
display(signal_b)


In [ ]:
# Print one-word verdicts from the executed tables.
# Signal A: confirmed when >=300-impression buckets have a real opportunity pool
# and their observed opportunity rate is higher than the <100 bucket.
a_low = signal_a.loc[signal_a["impression_bucket"].eq("<100"), "opportunity_rate"]
a_high = signal_a.loc[signal_a["impression_bucket"].isin(
    ["300-2,999", "3,000-29,999", "30,000+"]
), "opportunity_rate"].dropna()

if len(a_high) and len(a_low) and a_high.max() > a_low.iloc[0]:
    verdict_a = "CONFIRMED"
elif len(a_high) and len(a_low) and a_high.max() < a_low.iloc[0]:
    verdict_a = "OPPOSITE"
elif len(a_high) and len(a_low):
    verdict_a = "MIXED"
else:
    verdict_a = "FALSE"

if clicks:
    med = signal_b.set_index("position_bucket")["median_ctr_pct"]
    upper = med.reindex(["top_3", "page_1", "striking"]).dropna()
    deep = med.reindex(["page_3_5", "deep"]).dropna()
    if len(upper) and len(deep):
        if upper.median() > deep.median():
            verdict_b = "CONFIRMED"
        elif upper.median() < deep.median():
            verdict_b = "OPPOSITE"
        else:
            verdict_b = "MIXED"
    else:
        verdict_b = "FALSE"
else:
    # Without clicks, we cannot honestly test CTR-vs-position.
    verdict_b = "FALSE"

print("Signal A verdict:", verdict_a)
print("Signal B verdict:", verdict_b)


## 2. Encode ONE baseline rule

**Plain-language rule:** review pages that have at least **300 March impressions** and an average search position from **4 through 20**.

**Score:** March impressions for pages meeting both conditions; otherwise 0.

**One reason code:** `visible_position_opportunity`.

**Action label:** `review_refresh` for the opportunity group; `monitor` otherwise.

The score is transparent and unfitted. It uses only March decision-time fields.


In [ ]:
# Transparent, unfitted baseline score.
df["in_position_band"] = df["avg_position"].between(4, 20, inclusive="both").astype(int)
df["has_visibility"] = (df["impressions"] >= 300).astype(int)
df["score"] = (
    df["in_position_band"]
    * df["has_visibility"]
    * df["impressions"].clip(lower=0)
)

df["reason_code"] = df["score"].gt(0).map({
    True: "visible_position_opportunity",
    False: "insufficient_signal",
})
df["action"] = df["score"].gt(0).map({
    True: "review_refresh",
    False: "monitor",
})

queue = (
    df.sort_values(["score", "impressions"], ascending=False)
      .reset_index(drop=True)
)
queue["rank"] = queue.index + 1

out = queue[[
    "rank", "client_id", "content_id", "score", "reason_code",
    "action", "impressions", "avg_position"
]]

display(out.head(10))

os.makedirs("work/outputs", exist_ok=True)
out.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote work/outputs/baseline_action_score.csv with", len(out), "rows")


## 3. Top-10 skeptic review

For each top-ranked page, record:
- **Action** — what the rule recommends.
- **Why** — which rule conditions put it there.
- **What would make it wrong** — a concrete reason the human reviewer should reject or downgrade the recommendation.

These notes are deliberately skeptical: ranking is decision support, not proof that a refresh will work.


In [ ]:
top10 = queue.head(10).copy()

def skeptic(row):
    if row["action"] == "review_refresh":
        return (
            f"Action: review_refresh | "
            f"Why: {row['impressions']:.0f} March impressions and average position "
            f"{row['avg_position']:.1f} are inside the 4–20 opportunity band | "
            f"Wrong if impressions are temporary/noisy, query-level positions differ "
            f"from the average, the page already satisfies intent, or business priority "
            f"says not to change it."
        )
    return (
        "Action: monitor | Why: the page does not meet the visibility + position rule | "
        "Wrong if tracking/history is incomplete or the page is strategically important "
        "despite low measured visibility."
    )

top10["skeptic_review"] = top10.apply(skeptic, axis=1)
display(top10[[
    "rank", "action", "reason_code", "impressions",
    "avg_position", "skeptic_review"
]])
print("Top-10 review count:", len(top10))


## 4. Weak picks + leakage check

A **weak pick** is a row that technically satisfies the rule but is not obviously an editorial opportunity. Inspect at least one and keep the skeptical explanation.

**Leakage guard:** the rule must not use `trend_direction`, `trend_pct`, any future-month metric, or a label-derived field.


In [ ]:
eligible = queue[queue["score"] > 0].copy()

if len(eligible):
    # Show a few eligible rows with lower position within the 4–20 band.
    weak = eligible.sort_values(
        ["avg_position", "score"], ascending=[True, True]
    ).tail(min(3, len(eligible)))
    display(weak[[
        "rank", "content_id", "score", "impressions",
        "avg_position", "reason_code", "action"
    ]])
    print(
        "Weak-pick review: inspect these rows manually; "
        "a high score does not prove a refresh is correct."
    )
else:
    print(
        "No eligible picks in this slice — report the negative result "
        "rather than inventing picks."
    )

score_fields = ["impressions", "avg_position", "in_position_band", "has_visibility", "score"]
for forbidden in ["trend_direction", "trend_pct", "is_declining_label"]:
    assert forbidden not in score_fields, forbidden

print("Leakage guard passed: no label-derived or future-window field is used by the score.")


## Self-check

- [ ] Two visible bucket tables with `n`.
- [ ] Both signals are linked to session flags.
- [ ] Each signal has one printed verdict: `CONFIRMED`, `OPPOSITE`, `MIXED`, or `FALSE`.
- [ ] One transparent score, one reason code, one action label.
- [ ] `work/outputs/baseline_action_score.csv` is regenerated by the notebook.
- [ ] Top 10 each have action + why + what would make it wrong.
- [ ] At least one weak pick is surfaced, or the absence of eligible picks is reported honestly.
- [ ] No future-window or label-derived input enters the score.
- [ ] Run the notebook in Colab and commit the **executed** notebook before submission.

**Named limitation:** this baseline only knows March search visibility and average position. It does not know query intent, SERP features, business value, or whether a page is strategically important, so human review remains necessary.
